In [25]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize

# Half-lives (in seconds)
T_half_P = 1.687
T_half_D = 10.51 * 60
T_half_G = 91.8 * 60

# Decay constants
lambda_P = np.log(2) / T_half_P
lambda_D = np.log(2) / T_half_D
lambda_G = np.log(2) / T_half_G

# Time points
t = np.linspace(0, 3600, 3601)  # Simulate for 1 hour, including the endpoint

# Initial number of nuclei
N_P0 = 1000
N_D0 = 0
N_G0 = 0

# Placeholder function for the unknown addition rate
def unknown_addition_rate(t, Q):
    return Q if (t % 20 < 10) else 0  # Add parent nuclei every 20 seconds for 10 seconds

# Define the system of differential equations
def decay_chain(t, y, Q):
    N_P, N_D, N_G = y
    dN_P_dt = -lambda_P * N_P + unknown_addition_rate(t, Q)
    dN_D_dt = lambda_P * N_P - lambda_D * N_D
    dN_G_dt = lambda_D * N_D - lambda_G * N_G
    return [dN_P_dt, dN_D_dt, dN_G_dt]

# Simulate periodic background measurement
def background_measurement_func(Q):
    sol = solve_ivp(decay_chain, [0, t[-1]], [N_P0, N_D0, N_G0], t_eval=t, args=(Q,))
    N_P = sol.y[0]
    N_D = sol.y[1]
    N_G = sol.y[2]
    R_P = lambda_P * N_P
    R_D = lambda_D * N_D
    R_G = lambda_G * N_G
    R_total = R_P + R_D + R_G

    background_times = []
    background_measurement = []
    for idx, ti in enumerate(t):
        if (ti % 20 >= 10):  # Background measurement period
            background_times.append(ti)
            background_measurement.append(R_D[idx] + R_G[idx])

    background_times = np.array(background_times)
    background_measurement = np.array(background_measurement)
    if len(background_times) > 1:
        background_measurement_interp = np.interp(t, background_times, background_measurement)
    else:
        background_measurement_interp = np.zeros_like(t)

    print(f"R_total shape: {R_total.shape}")
    print(f"background_times shape: {background_times.shape}")
    print(f"background_measurement shape: {background_measurement.shape}")
    print(f"background_measurement_interp shape: {background_measurement_interp.shape}")

    return R_total, background_measurement_interp

# Observed total decay rates (assuming some noise in measurement)
Q_true = 10  # True value of the parent addition rate
R_total_observed, _ = background_measurement_func(Q_true)
R_total_observed += np.random.normal(0, 0.05, size=R_total_observed.shape)  # Adding some noise

print(f"R_total_observed shape: {R_total_observed.shape}")

# Define the objective function for optimization
def objective(Q):
    R_total, background_measurement_interp = background_measurement_func(Q)
    estimated_R_P = R_total - background_measurement_interp
    error = R_total_observed - R_total

    print(f"Q: {Q}")
    print(f"R_total shape: {R_total.shape}")
    print(f"background_measurement_interp shape: {background_measurement_interp.shape}")
    print(f"estimated_R_P shape: {estimated_R_P.shape}")
    print(f"error shape: {error.shape}")

    return np.sum(error**2)

# Estimate the parent addition rate using optimization
result = minimize(objective, x0=[5], bounds=[(0, 20)])
Q_estimated = result.x[0]
print(f"Estimated Parent Addition Rate: {Q_estimated}")

# Plot the results
R_total_estimated, background_measurement_interp = background_measurement_func(Q_estimated)
estimated_R_P = R_total_estimated - background_measurement_interp

plt.plot(t, R_total_observed, label='Observed Total Decay Rate')
plt.plot(t, R_total_estimated, '--', label='Estimated Total Decay Rate')
plt.plot(t, estimated_R_P, '--', label='Estimated Parent Decay Rate')
plt.xlabel('Time (seconds)')
plt.ylabel('Decay Rate')
plt.legend()
plt.show()


R_total shape: (3601,)
background_times shape: (1800,)
background_measurement shape: (1800,)
background_measurement_interp shape: (3601,)
R_total_observed shape: (3601,)


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.